# Quantized Retrieval-Augmented Medical VLM Diagnosis - Kaggle Full Test

Notebook này cấu hình một lượt test đầy đủ trên Kaggle cho dự án `medical_vlm_pipeline`:

1. Chuẩn bị workspace/source code.
2. Kiểm tra GPU, Python, PyTorch, CUDA.
3. Cài dependency.
4. Kiểm tra dataset IU Chest X-rays.
5. Compile source code.
6. Chạy demo pipeline nhanh.
7. Chạy training/evaluation full test `10` epoch.
8. Kiểm tra artifacts: checkpoint, metrics CSV/JSON, plot.

Khuyến nghị Kaggle: bật **GPU T4/P100/A100** và bật **Internet** nếu muốn tải pretrained weights từ Hugging Face/timm.

## 0. Central Test Configuration

Chỉnh các biến bên dưới trước khi chạy toàn notebook. Nếu source code đã nằm trong notebook folder, để `SOURCE_MODE = "current"`.

In [ ]:
from pathlib import Path

# Source options: "current", "github", or "kaggle_dataset"
SOURCE_MODE = "current"
GITHUB_REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/medical_vlm_pipeline.git"
KAGGLE_SOURCE_DIR = "/kaggle/input/medical-vlm-pipeline"

# Dataset config
DATASET_DIR = "/kaggle/input/iu-chest-x-rays-cleaned"
IMAGE_FOLDER = "resized_images/256"

# Full-test training config
EPOCHS = 10
BATCH_SIZE = 8
SYNTHETIC_CASES = 32
DEVICE = "auto"  # auto, cuda, mps, cpu

# Toggle stages
INSTALL_DEPS = True
RUN_DEMO = True
RUN_FULL_TRAIN = True
SKIP_PLOT = False  # keep False on Kaggle to generate training_curves.png

PROJECT_ROOT = Path("/kaggle/working/medical_vlm_pipeline")
SRC_DIR = PROJECT_ROOT / "src"
print("Config ready")

## 1. Prepare Workspace

Ba chế độ:

- `current`: dùng source đã có sẵn trong notebook/session.
- `github`: clone repo từ GitHub.
- `kaggle_dataset`: copy source đã upload dưới dạng Kaggle Dataset.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

if SOURCE_MODE == "github":
    os.chdir("/kaggle/working")
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", GITHUB_REPO_URL, str(PROJECT_ROOT)], check=True)
    os.chdir(SRC_DIR)
elif SOURCE_MODE == "kaggle_dataset":
    os.chdir("/kaggle/working")
    if not PROJECT_ROOT.exists():
        shutil.copytree(KAGGLE_SOURCE_DIR, PROJECT_ROOT)
    os.chdir(SRC_DIR)
else:
    # Try common locations in this order.
    candidates = [
        Path.cwd(),
        Path("/kaggle/working/medical_vlm_pipeline/src"),
        Path("/kaggle/working/src"),
    ]
    selected = None
    for c in candidates:
        if (c / "train_pipeline.py").exists() and (c / "medical_vlm_pipeline").exists():
            selected = c
            break
    if selected is None:
        raise FileNotFoundError(
            "Cannot find train_pipeline.py. Set SOURCE_MODE='github' or SOURCE_MODE='kaggle_dataset'."
        )
    SRC_DIR = selected.resolve()
    PROJECT_ROOT = SRC_DIR.parent.resolve()
    os.chdir(SRC_DIR)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)
print("Files:")
for path in sorted(SRC_DIR.iterdir()):
    print(" -", path.name)

## 2. Environment and Accelerator Check

Cell này xác nhận Kaggle đang nhận GPU hay chưa. Nếu `torch.cuda.is_available()` là `False`, hãy bật Accelerator trong Notebook Settings.

In [ ]:
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
else:
    print("WARNING: Kaggle GPU is not active. Training will fall back to CPU and run slower.")

## 3. Install Dependencies

Ưu tiên dùng `src/requirements.txt`. Nếu Kaggle gặp conflict dependency, cell fallback bên dưới cài tối thiểu các package cần cho full test.

In [ ]:
import subprocess
import sys

if INSTALL_DEPS:
    req = SRC_DIR / "requirements.txt"
    if req.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
    else:
        packages = [
            "torch", "torchvision", "timm", "transformers", "monai", "pydicom", "nibabel",
            "SimpleITK", "scikit-learn", "faiss-cpu", "pandas", "matplotlib", "rich", "tqdm",
            "nltk", "rouge-score", "evaluate", "bert-score", "grad-cam", "qdrant-client",
        ]
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
else:
    print("Dependency installation skipped by config.")

## 4. Verify Imports and Source Compilation

Nếu cell này lỗi, dừng lại xử lý dependency/source trước khi chạy train.

In [ ]:
import subprocess
import sys
from pathlib import Path

sys.path.insert(0, str(SRC_DIR))

import torch
import pandas as pd
import numpy as np
from medical_vlm_pipeline import PipelineConfig, MedicalVLMPipeline, MedicalCaseDataset, load_iu_chest_xray_cases

print("Imports OK")
subprocess.run([sys.executable, "-m", "compileall", "medical_vlm_pipeline", "train_pipeline.py", "demo_pipeline.py"], check=True)

## 5. Dataset Sanity Check

Notebook kỳ vọng dataset IU Chest X-rays cleaned có cấu trúc:

```text
/kaggle/input/iu-chest-x-rays-cleaned
  cleaned_dataset.csv
  resized_images/256/
```

Nếu không có dataset, training script sẽ tự chạy synthetic fallback để smoke test pipeline.

In [ ]:
import os
import pandas as pd
from pathlib import Path

csv_path = Path(DATASET_DIR) / "cleaned_dataset.csv"
images_dir = Path(DATASET_DIR) / IMAGE_FOLDER

print("CSV path:", csv_path)
print("Images dir:", images_dir)
print("CSV exists:", csv_path.exists())
print("Images dir exists:", images_dir.exists())

if csv_path.exists():
    df = pd.read_csv(csv_path)
    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    display(df.head(3))
    cases = load_iu_chest_xray_cases(csv_path, images_dir)
    print("Loaded cases:", len(cases))
    if cases:
        print("First case:", cases[0])
else:
    print("WARNING: Dataset not found. Full test will use synthetic fallback data.")

## 6. Quick End-to-End Demo

Chạy demo pipeline độc lập: synthetic cases, quantization, FAISS retrieval, diagnosis, generated report, InfoNCE check, explainability hook.

In [ ]:
import subprocess
import sys

if RUN_DEMO:
    subprocess.run([sys.executable, "demo_pipeline.py"], check=True)
else:
    print("Demo skipped by config.")

## 7. Full Training and Evaluation Test

Cell này chạy `train_pipeline.py` với cấu hình full test. Trên Kaggle GPU và dataset thật, command nên tự chọn `cuda` qua `--device auto`.

In [ ]:
import subprocess
import sys
from pathlib import Path

cmd = [
    sys.executable, "train_pipeline.py",
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--device", DEVICE,
    "--dataset-dir", DATASET_DIR,
    "--synthetic-cases", str(SYNTHETIC_CASES),
]
if SKIP_PLOT:
    cmd.append("--skip-plot")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

## 8. Artifact Inspection

Kiểm tra output sau train. Các file này được thiết kế để plot/phân tích lại về sau mà không cần chạy train lại:

- `run_config.json`: config, args, dataset stats.
- `environment.json`: Python/PyTorch/GPU runtime.
- `training_metrics.csv`: bảng epoch-level để plot loss/accuracy/F1/LR/duration.
- `epoch_metrics.json`: epoch-level đầy đủ, gồm per-class metrics và confusion matrix.
- `text_generation_metrics.json`: aggregate BLEU/ROUGE.
- `text_generation_samples.csv/json`: từng sample report generation + retrieval.
- `inference_report.json`: final diagnosis + step latency/shape/score.
- `run_summary.json`: summary cuối cùng gom các chỉ số quan trọng.

In [ ]:
from pathlib import Path
import json
import pandas as pd

root = SRC_DIR
report_dir = root / "report_kaggle"
artifact_paths = {
    "checkpoint": root / "best_model.pt",
    "run_config": report_dir / "run_config.json",
    "environment": report_dir / "environment.json",
    "training_metrics": report_dir / "training_metrics.csv",
    "epoch_metrics": report_dir / "epoch_metrics.json",
    "text_metrics": report_dir / "text_generation_metrics.json",
    "text_samples_csv": report_dir / "text_generation_samples.csv",
    "text_samples_json": report_dir / "text_generation_samples.json",
    "inference_report": report_dir / "inference_report.json",
    "run_summary": report_dir / "run_summary.json",
    "artifacts_manifest": report_dir / "artifacts_manifest.json",
    "training_curves": report_dir / "training_curves.png",
}

for name, path in artifact_paths.items():
    print(f"{name}: {path} | exists={path.exists()} | size={path.stat().st_size if path.exists() else 0}")

metrics_path = artifact_paths["training_metrics"]
if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
    display(metrics_df.tail())

text_samples_path = artifact_paths["text_samples_csv"]
if text_samples_path.exists():
    text_samples_df = pd.read_csv(text_samples_path)
    display(text_samples_df[["case_id", "label", "predicted_diagnosis", "confidence", "bleu_1", "rouge_l", "num_retrieved"]].head())

for key in ["text_metrics", "inference_report", "run_summary"]:
    path = artifact_paths[key]
    if path.exists():
        print(f"\n--- {key} ---")
        with open(path, "r", encoding="utf-8") as f:
            print(json.dumps(json.load(f), indent=2)[:4000])

## 9. Display Training Curves

Nếu `SKIP_PLOT=False`, plot sẽ hiển thị ở đây.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

plot_path = SRC_DIR / "report_kaggle" / "training_curves.png"
if plot_path.exists():
    display(Image(filename=str(plot_path)))
else:
    print("No training_curves.png found. Check SKIP_PLOT or Matplotlib errors.")

## 10. Expected Full-Test Success Criteria

Một lượt test đạt yêu cầu khi:

- Environment check thấy `CUDA available: True` trên Kaggle GPU.
- Source compile không lỗi.
- Dataset thật được load, hoặc synthetic fallback chạy thành công nếu chỉ smoke test.
- `train_pipeline.py` hoàn tất 10 epoch.
- Có `best_model.pt`.
- Có `report_kaggle/run_config.json` và `report_kaggle/environment.json`.
- Có `report_kaggle/training_metrics.csv` để plot loss/accuracy/F1/LR/duration.
- Có `report_kaggle/epoch_metrics.json` để plot per-class metrics/confusion matrix.
- Có `report_kaggle/text_generation_metrics.json` và `report_kaggle/text_generation_samples.csv/json`.
- Có `report_kaggle/inference_report.json` chứa step-by-step latency/shape/similarity scores.
- Có `report_kaggle/run_summary.json` và `report_kaggle/artifacts_manifest.json`.
- Có `report_kaggle/training_curves.png` nếu không bật `SKIP_PLOT`.